# PHẦN 1: DATA CLEANSING & FEATURE ENGINEERING

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# In tiêu đề câu bằng định dạng dễ nhìn
print("=" * 60)
print("CÂU 1: TẢI DỮ LIỆU VÀ HIỂN THỊ 10 DÒNG ĐẦU TIÊN")
print("=" * 60)

# Định nghĩa hàm tải dữ liệu từ tệp CSV
def load_data(file_path):
    df = pd.read_csv(file_path)  # Đọc file dữ liệu vào cấu trúc bảng DataFrame
    return df
# Thực hiện tải file dữ liệu thực tế trên máy của bạn
df = load_data("titanic_disaster.csv")
# ĐỂ DÒNG NÀY Ở CUỐI: Không bọc trong lệnh print() để Notebook tự render bảng HTML đẹp
df.head(10)

In [ ]:
print("=" * 60)
print("CÂU 2: THỐNG KÊ VÀ TRỰC QUAN HÓA DỮ LIỆU THIẾU")
print("=" * 60)

# Tính toán số lượng và tỷ lệ khuyết thiếu dữ liệu
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_table = pd.DataFrame(
    {"Số lượng thiếu": missing_count, "Tỷ lệ thiếu (%)": missing_percentage}
)

# Trực quan hóa bằng biểu đồ Heatmap
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Biểu đồ Heatmap thể hiện dữ liệu thiếu ban đầu")
plt.show()

# Hiển thị bảng thống kê dữ liệu thiếu ở dòng cuối
missing_table[missing_table["Số lượng thiếu"] > 0]

In [ ]:
print("=" * 60)
print("CÂU 3: TÁCH CỘT NAME THÀNH FIRSTNAME VÀ SECONDNAME RỒI XÓA CỘT GỐC")
print("=" * 60)

# Check if 'Name' column exists before processing
if 'Name' in df.columns:
    # Tách cột Name dựa vào dấu phẩy đầu tiên
    df[["firstName", "secondName"]] = df["Name"].str.split(",", n=1, expand=True)

    # Loại bỏ khoảng trắng thừa ở hai đầu chuỗi văn bản sau khi tách
    df["firstName"] = df["firstName"].str.strip()
    df["secondName"] = df["secondName"].str.strip()

    # Xóa bỏ hoàn toàn cột Name gốc ra khỏi bảng dữ liệu
    df.drop(columns=["Name"], inplace=True)

# Hiển thị kết quả dạng bảng đẹp ở dòng cuối, chỉ khi các cột mới đã được tạo
if 'firstName' in df.columns and 'secondName' in df.columns:
    print(df[["firstName", "secondName"]].head())
else:
    print("Các cột 'firstName' và 'secondName' không tồn tại, có thể cột 'Name' đã được xử lý hoặc không có sẵn.")

In [ ]:
print("=" * 60)
print("CÂU 4: RÚT GỌN KÍCH THƯỚC DỮ LIỆU TRÊN CỘT SEX")
print("=" * 60)

# Thay thế giá trị male -> M và female -> F
df["Sex"] = df["Sex"].map({"male": "M", "female": "F"})

# Hiển thị kết quả dạng bảng chứa cột Sex ở dòng cuối
df[["Sex"]].head()

In [ ]:
print("=" * 60)
print("CÂU 5: XỬ LÝ DỮ LIỆU THIẾU TRÊN BIẾN AGE THEO HẠNG VÉ")
print("=" * 60)

# --- Bước a: Vẽ Box plot khảo sát phân phối tuổi theo hạng vé ---
plt.figure(figsize=(8, 5))
sns.boxplot(x="Pclass", y="Age", data=df)
plt.title("Phân phối tuổi theo từng Hạng hành khách (Pclass)")
plt.show()

# --- Bước b: Điền khuyết dữ liệu Age bằng giá trị trung vị của từng Pclass ---
df["Age"] = df.groupby("Pclass")["Age"].transform(
    lambda x: x.fillna(x.median())
)

# Vẽ lại biểu đồ Heatmap để kiểm tra dữ liệu sau xử lý
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap="viridis")
plt.title("Biểu đồ Heatmap sau khi đã xử lý xong dữ liệu thiếu cột Age")
plt.show()

# Hiển thị kết quả bảng dữ liệu chứa cột Age đã làm sạch
df[["Pclass", "Age"]].head(10)

In [ ]:
print("=" * 60)
print("CÂU 6: XÂY DỰNG BIẾN SỐ AGEGROUP PHÂN NHÓM TUỔI")
print("=" * 60)

# Thiết lập khoảng cắt và nhãn tương ứng
age_bins = [0, 12, 18, 60, float("inf")]
age_labels = ["Kid", "Teen", "Adult", "Older"]

# Tiến hành cắt phân loại nhóm tuổi
df["Agegroup"] = pd.cut(
    df["Age"], bins=age_bins, labels=age_labels, include_lowest=True
)

# Xuất bảng đối chiếu độ tuổi và nhóm tuổi ở dòng cuối
df[["Age", "Agegroup"]].head(10)

In [ ]:
print("=" * 60)
print("CÂU 7: TRÍCH XUẤT ĐẶC TRƯNG DANH XƯNG (NAMEPREFIX)")
print("=" * 60)


# Hàm quét chuỗi để bóc tách danh xưng
def extract_prefix(name_string):
    for prefix in ["Mr.", "Mrs.", "Miss.", "Master."]:
        if prefix in name_string:
            return prefix.replace(".", "")
    return "Rare"


# Áp dụng hàm lên cột secondName để tạo đặc trưng mới
df["namePrefix"] = df["secondName"].apply(extract_prefix)

# Xuất kết quả dạng bảng ở dòng cuối
df[["secondName", "namePrefix"]].head()

In [ ]:
print("=" * 60)
print("CÂU 8: TÍNH TOÁN KÍCH THƯỚC GIA ĐÌNH (FAMILYSIZE)")
print("=" * 60)

# Áp dụng công thức tính quy mô gia đình
df["familySize"] = 1 + df["SibSp"] + df["Parch"]

# Xuất kết quả dạng bảng ở dòng cuối
df[["SibSp", "Parch", "familySize"]].head()

In [ ]:
print("=" * 60)
print("CÂU 9: TẠO ĐẶC TRƯNG ALONE (ĐI MỘT MÌNH HOẶC THEO NHÓM)")
print("=" * 60)

# Nếu kích thước gia đình bằng 1 thì Alone = 1, ngược lại là 0
df["Alone"] = np.where(df["familySize"] == 1, 1, 0)

# Xuất kết quả dạng bảng ở dòng cuối
df[["familySize", "Alone"]].head()

In [ ]:
print("=" * 60)
print("CÂU 10: TÁCH LOẠI CABIN VÀ XỬ LÝ KHUYẾT DỮ LIỆU")
print("=" * 60)

# Lấp đầy ô trống bằng chuỗi "Unknown"
df["Cabin"] = df["Cabin"].fillna("Unknown")

# Cắt lấy ký tự chữ cái đầu tiên đại diện cho tầng cabin
df["typeCabin"] = df["Cabin"].apply(
    lambda x: x[0] if x != "Unknown" else "Unknown"
)

# Hiển thị bảng tổng hợp kết quả của câu 10 ở dòng cuối cùng
df[["Cabin", "typeCabin"]].head(10)

# PHẦN 2: KHAI THÁC THÔNG TIN HỮU ÍCH – EDA

In [ ]:
print("=" * 60)
print("CÂU 12: TƯƠNG QUAN TỶ LỆ SỐNG SÓT VÀ THIỆT MẠNG THEO GIỚI TÍNH")
print("=" * 60)

# Khởi tạo khung chứa biểu đồ
plt.figure(figsize=(7, 5))

# Vẽ biểu đồ cột đếm số lượng (Countplot) phân tách theo giới tính và trạng thái sống sót
# Cột Sex (M/F), Hue='Survived' phân màu theo sống (1) / chết (0)
sns.countplot(x="Sex", hue="Survived", data=df, palette="Set2")

plt.title("Số lượng hành khách Sống sót / Thiệt mạng theo Giới tính")
plt.xlabel("Giới tính (Sex)")
plt.ylabel("Số lượng hành khách")
plt.legend(title="Trạng thái", labels=["Thiệt mạng (0)", "Sống sót (1)"])
plt.show()  # Lệnh bắt buộc để hiển thị biểu đồ trong Notebook

# Hiển thị thêm bảng số liệu thống kê chi tiết dạng bảng HTML đẹp
display(pd.crosstab(df["Sex"], df["Survived"], margins=True))

In [ ]:
print("=" * 60)
print("CÂU 13: THÔNG TIN HÀNH KHÁCH SỐNG SÓT TRÊN TỪNG HẠNG VÉ (PCLASS)")
print("=" * 60)

plt.figure(figsize=(7, 5))

# Vẽ biểu đồ cột biểu diễn tỷ lệ sống sót trung bình kèm thanh sai số (Barplot)
# Trục X là hạng vé, trục Y là Survived (tự động tính tỷ lệ phần trăm từ 0 đến 1)
sns.barplot(x="Pclass", y="Survived", data=df, palette="pastel", errorbar=None)

plt.title("Tỷ lệ sống sót trung bình theo Hạng hành khách (Pclass)")
plt.xlabel("Hạng hành khách (Pclass)")
plt.ylabel("Tỷ lệ sống sót trung bình")
plt.show()

# Hiển thị bảng số liệu phần trăm sống sót cụ thể của từng hạng vé
display(
    df.groupby("Pclass")["Survived"]
    .mean()
    .to_frame(name="Tỷ lệ sống sót trung bình")
)

In [ ]:
print("=" * 60)
print("CÂU 14: HÀNH KHÁCH SỐNG SÓT THEO GIỚI TÍNH VÀ THANG ĐO TUỔI TÁC")
print("=" * 60)

plt.figure(figsize=(10, 6))

# Sử dụng biểu đồ Catplot hoặc Barplot với cấu trúc:
# Trục X là nhóm tuổi (Agegroup từ câu 6), phân màu hue theo Giới tính (Sex), trục Y là tỷ lệ sống sót
sns.barplot(x="Agegroup", y="Survived", hue="Sex", data=df, palette="muted", errorbar=None)

plt.title("Tỷ lệ sống sót dựa trên Nhóm tuổi và Giới tính")
plt.xlabel("Nhóm tuổi (Agegroup)")
plt.ylabel("Tỷ lệ sống sót trung bình")
plt.legend(title="Giới tính")
plt.show()

# Hiển thị bảng pivot thống kê chi tiết
display(df.pivot_table(index="Agegroup", columns="Sex", values="Survived", aggfunc="mean"))

In [ ]:
print("=" * 60)
print("CÂU 15: XÁC SUẤT SỐNG SÓT DỰA TRÊN THÔNG TIN NHÓM ĐI CÙNG")
print("=" * 60)

# Thiết lập vẽ 2 biểu đồ nằm cạnh nhau để trực quan cả 2 khía cạnh: Quy mô gia đình và Đi một mình
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ 1: Tỷ lệ sống theo kích thước gia đình (familySize từ câu 8)
sns.lineplot(
    x="familySize", y="Survived", data=df, marker="o", ax=axes[0], color="g"
)
axes[0].set_title("Xác suất sống sót theo Kích thước gia đình")
axes[0].set_xlabel("Số thành viên trong gia đình")
axes[0].set_ylabel("Xác suất sống sót")

# Biểu đồ 2: Tỷ lệ sống sót giữa người đi một mình vs đi theo nhóm (Alone từ câu 9)
sns.barplot(x="Alone", y="Survived", data=df, palette="Blues", ax=axes[1], errorbar=None)
axes[1].set_title("Xác suất sống sót: Đi một mình (1) vs Đi theo nhóm (0)")
axes[1].set_xlabel("Trạng thái Alone")
axes[1].set_ylabel("Xác suất sống sót")

plt.tight_layout()
plt.show()

# Hiển thị bảng dữ liệu tóm tắt theo trạng thái Alone ở dòng cuối
display(
    df.groupby("Alone")["Survived"]
    .mean()
    .to_frame(name="Xác suất sống sót")
)

In [ ]:
print("=" * 60)
print("CÂU 16: XÁC SUẤT HÀNH KHÁCH SỐNG SÓT DỰA TRÊN GIÁ VÉ")
print("=" * 60)

plt.figure(figsize=(10, 5))

# Sử dụng biểu đồ phân phối mật độ (KDE Plot) để thấy sự khác biệt về phân bố giá vé
# giữa nhóm sống sót và nhóm thiệt mạng
sns.kdeplot(df[df["Survived"] == 1]["Fare"], label="Sống sót", fill=True, color="blue", alpha=0.4)
sns.kdeplot(df[df["Survived"] == 0]["Fare"], label="Thiệt mạng", fill=True, color="red", alpha=0.4)

# Giới hạn trục X lại một chút vì có một số vé hạng siêu sang rất cao làm biểu đồ bị lệch dài
plt.xlim(0, 200)

plt.title("Biểu đồ phân phối mật độ Giá vé (Fare) theo Trạng thái sống sót")
plt.xlabel("Giá vé (Fare)")
plt.ylabel("Mật độ phân bố")
plt.legend()
plt.show()

# Hiển thị bảng mô tả thông số giá vé trung bình của 2 nhóm
display(df.groupby("Survived")["Fare"].describe()[["mean", "min", "50%", "max"]])

In [ ]:
print("=" * 60)
print("CÂU 17: SỐ NGƯỜI SỐNG/CHẾT THEO HẠNG VÉ VÀ CẢNG CẬP BẾN")
print("=" * 60)

# Sử dụng FacetGrid hoặc Catplot của Seaborn để vẽ biểu đồ đa luồng cực kỳ mạnh mẽ
# Biểu đồ tách riêng từng Cảng cập bến (Embarked) thành các cột đồ thị nhỏ cạnh nhau
g = sns.catplot(
    x="Pclass",
    hue="Survived",
    col="Embarked",
    data=df,
    kind="count",
    palette="Set1",
    height=4,
    aspect=1,
)

# Cập nhật lại tiêu đề và các nhãn cho biểu đồ tổng thể
g.set_axis_labels("Hạng hành khách (Pclass)", "Số lượng hành khách")
g.set_titles("Cảng lên tàu: {col_name}")
g._legend.set_title("Trạng thái")

# Thay đổi tên nhãn trong Legend của đồ thị cho rõ ràng
new_labels = ["Thiệt mạng", "Sống sót"]
for t, l in zip(g._legend.texts, new_labels):
    t.set_text(l)

plt.show()

# Hiển thị bảng chéo 3 biến (Cảng x Hạng vé x Sống sót) dạng bảng HTML ở cuối cell
display(
    pd.crosstab(
        index=[df["Embarked"], df["Pclass"]],
        columns=df["Survived"],
        margins=True,
    )
)